In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("/kaggle/input/spring-2025-regression-competition/dataSP25.csv")
test_df = pd.read_csv("/kaggle/input/spring-2025-regression-competition/compSP25.csv")

2 - Insights and EDA¶

In [ ]:
df.head()

In [ ]:
df['price'].max()


In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df['host_id'].value_counts().head(100)


In [ ]:
df.drop(columns=['id', 'name', 'host_name'], inplace=True)
df.columns

In [ ]:
# Neues Feature: Hat eine Bewertung (1) oder nicht (0)
df['recent_reviewed'] = df['last_review'].notna().astype(int)

# Alte Spalte entfernen
df.drop('last_review', axis=1, inplace=True)

# Kontrolle
df['recent_reviewed'].value_counts()

In [ ]:
df['reviews_per_month'].fillna(0, inplace=True)

In [ ]:
df.isna().sum().sort_values(ascending = False)

In [ ]:
int(df.duplicated().sum())

In [ ]:
vis_num = ['price','minimum_nights', 'number_of_reviews','reviews_per_month'
           , 'calculated_host_listings_count','availability_365']


In [ ]:

sns.histplot(df['price'],kde=True)
plt.xlim(0, 1000)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df['price'], bins=50)
plt.show()


In [ ]:
df['price'].skew()

In [ ]:
df['price'] = np.log1p(df['price'])

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()


In [ ]:
df.isnull().sum()

In [ ]:
df['reviews_per_month'].fillna(0, inplace=True)


In [ ]:
df.count()

In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
sns.histplot(df['price'], bins=50, kde=True)
plt.title("Price Distribution")

plt.subplot(1,3,2)
sns.histplot(df['minimum_nights'], bins=50, kde=True)
plt.title("Minimum Nights Distribution")

plt.subplot(1,3,3)
sns.histplot(df['availability_365'], bins=50, kde=True)
plt.title("Availability 365 Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Neues Feature: Preis pro Nacht = Gesamtpreis / Mindestnächte
df['price_per_night'] = df['price'] / df['minimum_nights']

# Kontrolle
df[['price', 'minimum_nights', 'price_per_night']].head()

In [ ]:
# Wichtige Punkte in NYC
poi = {
    "times_square": (40.7580, -73.9855),
    "central_park": (40.7851, -73.9683),
    "jfk_airport": (40.6413, -73.7781),
    "liberty_statue": (40.6892, -74.0445),
    "brooklyn_bridge": (40.7061, -73.9969)
}

# Für jeden Punkt Distanz berechnen
for name, coords in poi.items():
    df[f"dist_{name}"] = np.hypot(
        df['latitude'] - coords[0],
        df['longitude'] - coords[1]
    )

# Falls du Lat/Lon nicht mehr brauchst, löschen:
df.drop(['latitude','longitude'], axis=1, inplace=True)

# Kontrolle
df[[col for col in df.columns if col.startswith("dist_")]].head()

In [ ]:
import seaborn as sns
sns.lmplot(data=df, x='dist_times_square', y='price_per_night', height=6, aspect=1.5)

🔤 One-Hot-Encoding¶
Kategorische Variablen (room_type, neighbourhood_group) in numerische Dummy-Features umwandeln.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

cat_cols = ['room_type', 'neighbourhood_group']
ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

encoded = ohe.fit_transform(df[cat_cols])
enc_cols = ohe.get_feature_names_out(cat_cols)

df = df.drop(columns=cat_cols).join(pd.DataFrame(encoded, columns=enc_cols, index=df.index))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numerische Spalten auswählen
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Korrelationsmatrix berechnen
corr = df[numeric_cols].corr()

# Heatmap zeichnen
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Korrelationsmatrix (Heatmap)")
plt.tight_layout()
plt.show()

In [ ]:
#Preis pro Nacht per_night
plt.figure(figsize=(6,4))
sns.scatterplot(data=df, x='price_per_night', y='price', alpha=0.4)
plt.title('Price vs. Price per Night')
plt.xlabel('Price per Night')
plt.ylabel('Total Price')
plt.tight_layout()
plt.show()

In [ ]:
# 📌 Features auswählen (nur vorhandene Spalten!)
feature_cols = [
    'room_type_Private room',
    'room_type_Shared room',
    'minimum_nights',
    'neighbourhood_group_Queens',
    'dist_times_square',
    'dist_central_park',
    'dist_jfk_airport',
    'dist_liberty_statue',
    'dist_brooklyn_bridge'
]

# 🔀 Features (X) und Zielvariable (y) definieren
X = df[feature_cols]
y = df['price_per_night']   # Ziel: Preis pro Nacht

# ✅ Kontrolle
X.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import numpy as np

# 📊 Trainings-/Test-Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 🤖 XGBoost-Regressionsmodell erstellen
model = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

# 🔧 Modell trainieren
model.fit(X_train, y_train)

# 🔮 Vorhersagen für Train- und Testdaten
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# 📈 Modellbewertung
print("Train R²:", r2_score(y_train, train_pred))
print("Test  R²:", r2_score(y_test, test_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, test_pred)))

In [ ]:
# 📂 Testdaten 

# ID zur späteren Submission behalten
test_ids = test_df["id"]

# ✅ Korrekt: price_per_night wird NICHT als Feature benutzt.
#    Unser Ziel (y) war price_per_night – im Test soll es vorhergesagt werden, nicht berechnet.

# 📍 Distanz-Features berechnen (wie im Training)
poi = {
    "times_square": (40.7580, -73.9855),
    "central_park": (40.7851, -73.9683),
    "jfk_airport": (40.6413, -73.7781),
    "liberty_statue": (40.6892, -74.0445),
    "brooklyn_bridge": (40.7061, -73.9969)
}

for name, coords in poi.items():
    test_df[f"dist_{name}"] = np.hypot(
        test_df['latitude'] - coords[0],
        test_df['longitude'] - coords[1]
    )

# Alte Koordinaten-Spalten können raus
test_df = test_df.drop(columns=['latitude','longitude'])

# 🏷️ Kategorische Features (wie beim Training)
cat_cols = ['room_type', 'neighbourhood_group']

# One-Hot-Encoding
test_df_encoded = pd.get_dummies(test_df, columns=cat_cols, drop_first=True)

# ⚖️ Sicherstellen, dass Features identisch zu Training sind
missing_cols = set(X.columns) - set(test_df_encoded.columns)
for col in missing_cols:
    test_df_encoded[col] = 0
test_df_encoded = test_df_encoded[X.columns]  # gleiche Reihenfolge

# 🔮 Vorhersagen
predictions = model.predict(test_df_encoded)



In [ ]:
# Negative Vorhersagen auf 0 setzen (keine Unterkunft ist < 0 $)
predictions = np.clip(predictions, 0, None)

print("Neue Min:", predictions.min())
print("Neue Max:", predictions.max())


In [ ]:
# exakt die Trainingsspalten (zur Sicherheit direkt aus X nehmen)
cols = list(X.columns)

# 1 Zeile mit Nullen, dann gezielt setzen
row = pd.DataFrame({c: [0.0] for c in cols})

# ---- Werte setzen (Beispiel) ----
# Zimmertyp: genau einer aktiv; Entire home/apt = beide 0
row.loc[0, 'room_type_Private room'] = 1   # Private room
row.loc[0, 'room_type_Shared room']  = 0   # nicht Shared

row.loc[0, 'minimum_nights'] = 3
row.loc[0, 'neighbourhood_group_Queens'] = 1   # 1=Queens, sonst 0

# Distanzen (Beispielwerte)
row.loc[0, 'dist_times_square']     = 0.08
row.loc[0, 'dist_central_park']     = 0.12
row.loc[0, 'dist_jfk_airport']      = 0.25
row.loc[0, 'dist_liberty_statue']   = 0.18
row.loc[0, 'dist_brooklyn_bridge']  = 0.10

# Vorhersage
pred = model.predict(row)[0]
print(f"Vorhergesagter Preis pro Nacht: ${pred:.2f}")

In [ ]:
import joblib

#modell speichern 
joblib.dump(model, "airbnb_model.pkl")

In [ ]:
# Negative Vorhersagen auf 0 setzen (keine Unterkunft ist < 0 $)
predictions = np.clip(predictions, 0, None)

print("Neue Min:", predictions.min())
print("Neue Max:", predictions.max())


In [ ]:
# 📄 Submission-Datei erstellen (nach Clipping)
submission_df = pd.DataFrame({
    'id': test_ids,
    'price_per_night': predictions
})

# CSV speichern
submission_df.to_csv("submissionSP25.csv", index=False)

# Kontrolle: Ersten Einträge anzeigen
submission_df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(submission_df['price_per_night'], bins=50, kde=True)
plt.title("Verteilung der vorhergesagten Preise pro Nacht")
plt.xlabel("Preis pro Nacht ($)")
plt.ylabel("Anzahl Listings")
plt.show()
